# AndinaLog 03B | Productos | Diagnóstico v2

Contrato didáctico: datos originales visibles, decisiones trazables y tres salidas CSV.


In [ ]:
from pathlib import Path
import hashlib
import sys
import pandas as pd

ENTORNO="auto"  # auto, local, drive
RUTA_PROYECTO_DRIVE="/content/drive/MyDrive/GIAD"
VERSION_DIAGNOSTICO="GIAD-M3-S4-PRODUCTOS-diagnostico-didactico-v2"
COLUMNAS_BRONZE=["producto_id","nombre_producto","categoria_logistica",
    "temperatura_conservacion_requerida_c","tolerancia_temperatura_c",
    "precio_unitario_bob","costo_unitario_bob"]
PARES_TERMICOS={"Fresco":(4.0,2.0),"Congelado":(-18.0,2.0),"Seco":(20.0,5.0)}

def encontrar_raiz():
    if ENTORNO=="drive" or (ENTORNO=="auto" and "google.colab" in sys.modules):
        from google.colab import drive
        drive.mount("/content/drive")
        raiz=Path(RUTA_PROYECTO_DRIVE)
        if not (raiz/"datasets/AndinaLog_03B_Bronce/andinalog_productos.csv").is_file():
            raise FileNotFoundError(raiz)
        return raiz
    for carpeta in [Path.cwd(),*Path.cwd().parents]:
        if (carpeta/"datasets/AndinaLog_03B_Bronce/andinalog_productos.csv").is_file():
            return carpeta
    raise FileNotFoundError("No se encontró la raíz de practicasNotebookColab")

RAIZ=encontrar_raiz()
RUTA_BRONZE=RAIZ/"datasets/AndinaLog_03B_Bronce/andinalog_productos.csv"
SALIDAS=RAIZ/"proyecto-integrador/01_diagnostico/andinalog_productos/salidas"
HASH_BRONZE=hashlib.sha256(RUTA_BRONZE.read_bytes()).hexdigest()
bronze=pd.read_csv(RUTA_BRONZE,dtype="string",encoding="utf-8-sig",keep_default_na=False)
if list(bronze.columns)!=COLUMNAS_BRONZE:
    raise ValueError(f"Esquema Bronze inesperado: {list(bronze.columns)}")
df=bronze.copy(deep=True)
df.insert(0,"fila_bronze",range(1,len(df)+1))
print("Filas Bronze:",len(df))


## Reglas técnicas: completitud, formato, dominios y unicidad

Una copia exacta posterior se distingue de una misma clave con atributos contradictorios. También se señala una colisión que aparecería al quitar espacios y pasar el ID a mayúsculas. El diagnóstico conserva todos los registros.


In [ ]:
PRIORIDAD={"OK":0,"NO_EVALUABLE":1,"REVISAR":2,"CRITICO":3}
for c in COLUMNAS_BRONZE:
    df[f"{c}_estado"]="OK"
    df[f"{c}_motivo"]=""

def marcar(columna,mascara,estado,motivo):
    mascara=pd.Series(mascara,index=df.index).fillna(False).astype(bool)
    e,m=f"{columna}_estado",f"{columna}_motivo"
    subir=mascara & df[e].map(PRIORIDAD).lt(PRIORIDAD[estado])
    df.loc[subir,e]=estado
    previo=df.loc[mascara,m]
    df.loc[mascara,m]=previo.where(previo.eq(""),previo+"; ")+motivo

def texto(c): return df[c].str.strip()
for c in COLUMNAS_BRONZE:
    marcar(c,texto(c).eq(""),"CRITICO","Valor faltante")

id_original=df["producto_id"]
id_limpio=id_original.str.strip()
id_normal=id_limpio.str.upper()
formato=id_original.str.fullmatch(r"PROD-\d{3}").fillna(False)
marcar("producto_id",id_limpio.ne("") & ~formato,"CRITICO","Formato esperado: PROD-###")

firma=pd.util.hash_pandas_object(df[COLUMNAS_BRONZE],index=False)
variantes_exactas=firma.groupby(id_limpio,dropna=False).transform("nunique")
conflicto_exacto=id_limpio.ne("") & id_limpio.duplicated(keep=False) & variantes_exactas.gt(1)
copia=df.duplicated(COLUMNAS_BRONZE,keep="first") & ~conflicto_exacto
marcar("producto_id",conflicto_exacto,"CRITICO","Mismo ID con atributos contradictorios")
marcar("producto_id",copia,"CRITICO","Copia exacta posterior")

# Analizar la clave normalizada sin alterar el Bronze.
atributos_normalizados=df[COLUMNAS_BRONZE[1:]].copy()
firma_atributos=pd.util.hash_pandas_object(atributos_normalizados,index=False)
variantes_normalizadas=firma_atributos.groupby(id_normal,dropna=False).transform("nunique")
colision=(id_normal.ne("") & id_normal.duplicated(keep=False) &
    ~id_limpio.duplicated(keep=False))
marcar("producto_id",colision & variantes_normalizadas.eq(1),"REVISAR",
       "ID equivalente a otro registro después de normalizar")
marcar("producto_id",colision & variantes_normalizadas.gt(1),"CRITICO",
       "ID normalizado colisiona con atributos distintos")

categoria=texto("categoria_logistica")
marcar("categoria_logistica",categoria.ne("") & ~categoria.isin(PARES_TERMICOS),
       "CRITICO","Categoría fuera de Fresco, Congelado o Seco")


## Reglas de temperatura, tolerancia y economía

El sufijo `_c` indica Celsius. Objetivo y tolerancia se evalúan numéricamente; una temperatura objetivo negativa es válida para congelados. Se comprueba la coherencia contra los pares observados del dataset solo cuando la categoría es reconocida. El diagnóstico no rellena objetivos faltantes.


In [ ]:
numeros={}
for c in ["temperatura_conservacion_requerida_c","tolerancia_temperatura_c",
          "precio_unitario_bob","costo_unitario_bob"]:
    n=pd.to_numeric(texto(c),errors="coerce")
    numeros[c]=n
    marcar(c,texto(c).ne("") & n.isna(),"CRITICO","Valor no numérico")

temp=numeros["temperatura_conservacion_requerida_c"]
tol=numeros["tolerancia_temperatura_c"]
precio=numeros["precio_unitario_bob"]
costo=numeros["costo_unitario_bob"]
marcar("tolerancia_temperatura_c",tol.notna() & tol.le(0),"CRITICO","Tolerancia debe ser mayor que 0 °C")
for c,n in [("precio_unitario_bob",precio),("costo_unitario_bob",costo)]:
    marcar(c,n.notna() & n.le(0),"CRITICO","Valor debe ser mayor que 0 BOB")

objetivo_esperado=categoria.map({k:v[0] for k,v in PARES_TERMICOS.items()})
tolerancia_esperada=categoria.map({k:v[1] for k,v in PARES_TERMICOS.items()})
marcar("temperatura_conservacion_requerida_c",temp.notna() & objetivo_esperado.notna() &
       temp.sub(objetivo_esperado).abs().gt(0.01),"REVISAR",
       "Objetivo térmico distinto del patrón observado para la categoría")
marcar("tolerancia_temperatura_c",tol.notna() & tolerancia_esperada.notna() &
       tol.sub(tolerancia_esperada).abs().gt(0.01),"REVISAR",
       "Tolerancia distinta del patrón observado para la categoría")
marcar("temperatura_conservacion_requerida_c",categoria.ne("") &
       ~categoria.isin(PARES_TERMICOS) & temp.notna(),"NO_EVALUABLE",
       "No se valida coherencia térmica con categoría no reconocida")
marcar("tolerancia_temperatura_c",categoria.ne("") &
       ~categoria.isin(PARES_TERMICOS) & tol.notna(),"NO_EVALUABLE",
       "No se valida coherencia térmica con categoría no reconocida")

marcar("precio_unitario_bob",precio.notna() & costo.notna() & precio.lt(costo),
       "REVISAR","Precio menor que costo; verificar decisión comercial")
print("Reglas aplicadas; Bronze intacto")


## Resumen, comprobaciones y exportación

Solo `CRITICO` envía una fila a cuarentena diagnóstica. `REVISAR` y `NO_EVALUABLE` conservan la fila en el diagnosticado para tratamiento o revisión contextual. La cuarentena no se concatena con el diagnosticado.


In [ ]:
# Public diagnostic contract: one quarantine flag and one reason per Bronze field.
for campo in COLUMNAS_BRONZE:
    df[f"{campo}_en_cuarentena"] = df[f"{campo}_estado"].eq("CRITICO")
df["en_cuarentena"] = df[[f"{c}_en_cuarentena" for c in COLUMNAS_BRONZE]].any(axis=1)
columnas_publicas = ["fila_bronze", *COLUMNAS_BRONZE]
columnas_publicas += [x for c in COLUMNAS_BRONZE for x in (f"{c}_en_cuarentena", f"{c}_motivo")]
columnas_publicas += ["en_cuarentena"]
diagnosticado = df[columnas_publicas].copy()
cuarentena = diagnosticado.loc[diagnosticado["en_cuarentena"]].copy()
conteos = {"filas_bronze":len(bronze), "filas_diagnosticadas":len(diagnosticado),
           "filas_cuarentena":len(cuarentena), "filas_con_observaciones":int(df[[f"{c}_motivo" for c in COLUMNAS_BRONZE]].ne("").any(axis=1).sum())}
for campo in COLUMNAS_BRONZE:
    conteos[f"cuarentena_{campo}"] = int(diagnosticado[f"{campo}_en_cuarentena"].sum())
reporte_calidad = pd.DataFrame([{"metrica":k,"valor":v} for k,v in conteos.items()])
assert len(diagnosticado)==len(bronze)
assert diagnosticado["en_cuarentena"].equals(diagnosticado[[f"{c}_en_cuarentena" for c in COLUMNAS_BRONZE]].any(axis=1))
assert len(cuarentena)==int(diagnosticado["en_cuarentena"].sum())
pd.testing.assert_frame_equal(diagnosticado[COLUMNAS_BRONZE],bronze)
SALIDAS.mkdir(parents=True,exist_ok=True)
base="andinalog_productos_didactico_v2_"
diagnosticado.to_csv(SALIDAS/(base+"diagnosticado.csv"),index=False,encoding="utf-8-sig")
cuarentena.to_csv(SALIDAS/(base+"cuarentena.csv"),index=False,encoding="utf-8-sig")
reporte_calidad.to_csv(SALIDAS/(base+"reporte_calidad.csv"),index=False,encoding="utf-8-sig")
print(conteos)
display(diagnosticado.tail(5))
